cell 1

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix
)

os.makedirs('../eda', exist_ok=True)

RANDOM_SEED = 42
print('Imports done.')

Imports done.


cell 2

In [2]:
regressor         = joblib.load('../models/grade_predictor.pkl')
scaler            = joblib.load('../models/feature_scaler.pkl')
selected_features = joblib.load('../models/feature_list.pkl')

df = pd.read_csv('../data/features/features_selectionv2.csv')

print(f'Dataset shape     : {df.shape}')
print(f'Features loaded   : {len(selected_features)}')
print(f'Features          : {selected_features}')

Dataset shape     : (5000, 9)
Features loaded   : 7
Features          : ['attendance_percentage', 'midterm_score', 'historical_gpa', 'study_hours_per_week', 'subject_difficulty_score', 'ca_avg', 'rule_risk_score']


cell 3

In [3]:
X     = df[selected_features]
y_reg = df['final_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg,
    test_size=0.2,
    random_state=RANDOM_SEED
)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

y_pred = np.clip(regressor.predict(X_test_scaled), 0, 100)

print(f'Test set size : {len(y_test):,} rows')
print(f'Pred range    : {y_pred.min():.2f} – {y_pred.max():.2f}')

Test set size : 1,000 rows
Pred range    : 44.88 – 87.85


cell 4: Core Regression Metrics

In [4]:
mae       = mean_absolute_error(y_test, y_pred)
rmse      = np.sqrt(mean_squared_error(y_test, y_pred))
r2        = r2_score(y_test, y_pred)
mape      = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
residuals = y_test.values - y_pred

print('── Regression Metrics — Test Set ──')
print(f'  MAE  : {mae:.4f}   {"✅ OK" if mae  < 8    else "⚠️  Above target"} (target < 8)')
print(f'  RMSE : {rmse:.4f}  {"✅ OK" if rmse < 12   else "⚠️  Above target"} (target < 12)')
print(f'  R²   : {r2:.4f}   {"✅ OK" if r2   >= 0.80 else "⚠️  Below target"} (target ≥ 0.80)')
print(f'  MAPE : {mape:.2f}%')
print()
print('── Residual Summary ──')
print(f'  Mean residual    : {residuals.mean():.4f}  (closer to 0 = less bias)')
print(f'  Std residual     : {residuals.std():.4f}')
print(f'  Max overpredict  : {residuals.min():.4f}')
print(f'  Max underpredict : {residuals.max():.4f}')

── Regression Metrics — Test Set ──
  MAE  : 2.5951   ✅ OK (target < 8)
  RMSE : 3.2475  ✅ OK (target < 12)
  R²   : 0.8437   ✅ OK (target ≥ 0.80)
  MAPE : 4.01%

── Residual Summary ──
  Mean residual    : 0.0829  (closer to 0 = less bias)
  Std residual     : 3.2464
  Max overpredict  : -12.4623
  Max underpredict : 10.5986


 cell 5 Plot 1: Predicted vs Actual Scatter

In [5]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_test, y_pred, alpha=0.35, s=18,
           color='steelblue', label='Predictions')

min_val = min(y_test.min(), y_pred.min()) - 2
max_val = max(y_test.max(), y_pred.max()) + 2
ax.plot([min_val, max_val], [min_val, max_val],
        color='red', linestyle='--', linewidth=1.5,
        label='Perfect Prediction')

ax.set_xlabel('Actual Final Score',    fontsize=12)
ax.set_ylabel('Predicted Final Score', fontsize=12)
ax.set_title('Predicted vs Actual Final Score',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.text(0.05, 0.92,
        f'R² = {r2:.4f}\nMAE = {mae:.4f}\nRMSE = {rmse:.4f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../eda/eval_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/eval_predicted_vs_actual.png')

Plot saved → eda/eval_predicted_vs_actual.png


 cell 6: Plot 2: Residual Analysis

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Residual Analysis', fontsize=14, fontweight='bold')

# Left — residual histogram
axes[0].hist(residuals, bins=40, color='steelblue',
             edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='red', linestyle='--',
                linewidth=1.5, label='Zero residual')
axes[0].axvline(residuals.mean(), color='orange', linestyle='-',
                linewidth=1.5, label=f'Mean = {residuals.mean():.3f}')
axes[0].set_xlabel('Residual (Actual − Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')
axes[0].legend()

# Right — residuals vs predicted
axes[1].scatter(y_pred, residuals, alpha=0.3, s=15, color='seagreen')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted Score')

plt.tight_layout()
plt.savefig('../eda/eval_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/eval_residuals.png')

Plot saved → eda/eval_residuals.png


 cell 7: Plot 3: XGBoost Feature Importance

In [7]:
importance = pd.Series(
    regressor.feature_importances_,
    index=selected_features
).sort_values(ascending=True)

colors = [
    '#d73027' if v == importance.max() else
    '#fc8d59' if v >= importance.quantile(0.75) else
    '#91bfdb'
    for v in importance.values
]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance.index, importance.values,
               color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, importance.values):
    ax.text(bar.get_width() + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance Score')
ax.set_title('XGBoost Regressor — Feature Importance (Trained Model)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, importance.max() * 1.18)

plt.tight_layout()
plt.savefig('../eda/eval_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Plot saved → eda/eval_feature_importance.png')
print()
print('── Feature Importance Ranking ──')
for feat, val in importance.sort_values(ascending=False).items():
    print(f'  {feat:<30}: {val:.4f}')

Plot saved → eda/eval_feature_importance.png

── Feature Importance Ranking ──
  ca_avg                        : 0.5740
  rule_risk_score               : 0.2641
  midterm_score                 : 0.0504
  study_hours_per_week          : 0.0346
  subject_difficulty_score      : 0.0342
  attendance_percentage         : 0.0267
  historical_gpa                : 0.0159


cell 8: Plot 4: Error by Grade Band

In [8]:
def assign_grade_band(score):
    if score >= 90:   return 'A+ (90-100)'
    elif score >= 80: return 'A  (80-89)'
    elif score >= 70: return 'B+ (70-79)'
    elif score >= 60: return 'B  (60-69)'
    elif score >= 50: return 'C+ (50-59)'
    elif score >= 40: return 'C  (40-49)'
    elif score >= 30: return 'D+ (30-39)'
    elif score >= 20: return 'D  (20-29)'
    else:             return 'E  (0-19)'

eval_df = pd.DataFrame({
    'actual'   : y_test.values,
    'predicted': y_pred,
    'residual' : residuals,
    'abs_error': np.abs(residuals)
})
eval_df['grade_band'] = eval_df['actual'].apply(assign_grade_band)

band_mae = eval_df.groupby('grade_band')['abs_error'].agg(['mean', 'count'])
band_mae.columns = ['MAE', 'Count']
band_mae = band_mae.sort_index()

print('── MAE by Grade Band ──')
print(f"{'Grade Band':<15} {'MAE':>8} {'Count':>8}")
print('-' * 35)
for band, row in band_mae.iterrows():
    print(f"{band:<15} {row['MAE']:>8.4f} {int(row['Count']):>8}")

fig, ax = plt.subplots(figsize=(11, 5))
band_plot = band_mae[band_mae['Count'] >= 5]
bars = ax.bar(band_plot.index, band_plot['MAE'],
              color='steelblue', edgecolor='white', alpha=0.85)
ax.axhline(mae, color='red', linestyle='--',
           linewidth=1.5, label=f'Overall MAE = {mae:.4f}')

for bar, (_, row) in zip(bars, band_plot.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"n={int(row['Count'])}",
            ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Grade Band (Actual Score)')
ax.set_ylabel('Mean Absolute Error')
ax.set_title('Prediction Error by Grade Band', fontweight='bold')
ax.legend()
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('../eda/eval_error_by_grade_band.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/eval_error_by_grade_band.png')

── MAE by Grade Band ──
Grade Band           MAE    Count
-----------------------------------
A  (80-89)        2.7866       48
B  (60-69)        2.4508      461
B+ (70-79)        2.7143      253
C  (40-49)        3.6877       25
C+ (50-59)        2.5947      213
Plot saved → eda/eval_error_by_grade_band.png


Plot 5: Risk Level & Grade Distribution

In [9]:
def derive_failure_probability(score):
    return float(np.clip((60 - score) / 20 + 0.5, 0.0, 1.0))

def derive_risk_level(prob):
    if prob >= 0.75:   return 'CRITICAL'
    elif prob >= 0.55: return 'HIGH'
    elif prob >= 0.35: return 'MEDIUM'
    else:              return 'LOW'

def derive_grade(score):
    if score >= 90:   return 'A+'
    elif score >= 80: return 'A'
    elif score >= 70: return 'B+'
    elif score >= 60: return 'B'
    elif score >= 50: return 'C+'
    elif score >= 40: return 'C'
    elif score >= 30: return 'D+'
    elif score >= 20: return 'D'
    else:             return 'E'

eval_df['fail_prob']  = eval_df['predicted'].apply(derive_failure_probability)
eval_df['risk_level'] = eval_df['fail_prob'].apply(derive_risk_level)
eval_df['pred_grade'] = eval_df['predicted'].apply(derive_grade)

risk_order  = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
risk_counts = eval_df['risk_level'].value_counts().reindex(risk_order, fill_value=0)
risk_colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']

print('── Risk Level Distribution (Test Set) ──')
for level, count in risk_counts.items():
    pct = count / len(eval_df) * 100
    print(f'  {level:<10}: {count:>4}  ({pct:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Risk Level & Grade Distribution — Test Set',
             fontsize=13, fontweight='bold')

bars = axes[0].bar(risk_counts.index, risk_counts.values,
                   color=risk_colors, edgecolor='white')
for bar, val in zip(bars, risk_counts.values):
    pct = val / len(eval_df) * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1,
                 f'{val}\n({pct:.1f}%)',
                 ha='center', va='bottom', fontsize=10)
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Risk Level Distribution')

grade_order  = ['A+','A','B+','B','C+','C','D+','D','E']
grade_counts = eval_df['pred_grade'].value_counts().reindex(grade_order, fill_value=0)
grade_colors = ['#1a9850','#66bd63','#a6d96a','#d9ef8b',
                '#fee08b','#fdae61','#f46d43','#d73027','#a50026']
bars2 = axes[1].bar(grade_counts.index, grade_counts.values,
                    color=grade_colors, edgecolor='white')
for bar, val in zip(bars2, grade_counts.values):
    if val > 0:
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.5,
                     str(val), ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Predicted Grade')
axes[1].set_ylabel('Number of Students')
axes[1].set_title('Predicted Grade Distribution')

plt.tight_layout()
plt.savefig('../eda/eval_risk_and_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/eval_risk_and_grade_distribution.png')

── Risk Level Distribution (Test Set) ──
  LOW       :  633  (63.3%)
  MEDIUM    :  183  (18.3%)
  HIGH      :  133  (13.3%)
  CRITICAL  :   51  (5.1%)
Plot saved → eda/eval_risk_and_grade_distribution.png


 Plot 6: Grade Agreement Heatmap

In [10]:
eval_df['actual_grade'] = eval_df['actual'].apply(derive_grade)

grade_order_list = ['E','D','D+','C','C+','B','B+','A','A+']
exact_match      = (eval_df['pred_grade'] == eval_df['actual_grade']).mean()

adjacent_match = 0
for _, row in eval_df.iterrows():
    ai = grade_order_list.index(row['actual_grade']) if row['actual_grade'] in grade_order_list else -1
    pi = grade_order_list.index(row['pred_grade'])   if row['pred_grade']   in grade_order_list else -1
    if abs(ai - pi) <= 1:
        adjacent_match += 1
adjacent_match /= len(eval_df)

print('── Grade Prediction Agreement ──')
print(f'  Exact match      : {exact_match:.1%}')
print(f'  Within 1 band    : {adjacent_match:.1%}')

present_grades = sorted(
    set(eval_df['actual_grade'].tolist() + eval_df['pred_grade'].tolist()),
    key=lambda g: grade_order_list.index(g) if g in grade_order_list else 99
)
cm = confusion_matrix(
    eval_df['actual_grade'],
    eval_df['pred_grade'],
    labels=present_grades
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=present_grades,
            yticklabels=present_grades,
            cmap='Blues', linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted Grade', fontsize=12)
ax.set_ylabel('Actual Grade',    fontsize=12)
ax.set_title(
    f'Grade Prediction Agreement\n'
    f'Exact match: {exact_match:.1%}  |  Within 1 band: {adjacent_match:.1%}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../eda/eval_grade_agreement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/eval_grade_agreement.png')

── Grade Prediction Agreement ──
  Exact match      : 72.3%
  Within 1 band    : 100.0%
Plot saved → eda/eval_grade_agreement.png


Early Warning Threshold Analysis

In [11]:
eval_df['actually_failed'] = (eval_df['actual'] < 60).astype(int)

print('── Early Warning Threshold Analysis ──')
print(f"{'Risk Level':<12} {'Total':>7} {'Actually Failed':>16} {'Precision':>10}")
print('-' * 50)

for level in ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']:
    subset    = eval_df[eval_df['risk_level'] == level]
    total     = len(subset)
    failed    = subset['actually_failed'].sum()
    precision = failed / total if total > 0 else 0
    print(f'{level:<12} {total:>7} {failed:>16} {precision:>10.1%}')

print()
total_actual_failures   = eval_df['actually_failed'].sum()
caught_in_high_critical = eval_df[
    (eval_df['risk_level'].isin(['HIGH', 'CRITICAL'])) &
    (eval_df['actually_failed'] == 1)
].shape[0]
recall = caught_in_high_critical / total_actual_failures if total_actual_failures > 0 else 0

print(f'Total actual failures in test set : {total_actual_failures}')
print(f'Caught by HIGH + CRITICAL alerts  : {caught_in_high_critical}')
print(f'Early warning recall              : {recall:.1%}')

── Early Warning Threshold Analysis ──
Risk Level     Total  Actually Failed  Precision
--------------------------------------------------
LOW              633               18       2.8%
MEDIUM           183               68      37.2%
HIGH             133              102      76.7%
CRITICAL          51               50      98.0%

Total actual failures in test set : 238
Caught by HIGH + CRITICAL alerts  : 152
Early warning recall              : 63.9%


Confusion metrics

In [14]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ── Define grade order ────────────────────────────────────────────────────────
grade_order_list = ['E', 'D', 'D+', 'C', 'C+', 'B', 'B+', 'A', 'A+']

# Only include grades that actually appear in test set
present_grades = sorted(
    set(eval_df['actual_grade'].tolist() + eval_df['pred_grade'].tolist()),
    key=lambda g: grade_order_list.index(g) if g in grade_order_list else 99
)

# ── Compute confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(
    eval_df['actual_grade'],
    eval_df['pred_grade'],
    labels=present_grades
)

# ── Normalized confusion matrix (row-wise = recall per class) ─────────────────
cm_normalized = cm.astype(float)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm_normalized, row_sums,
    where=row_sums != 0
)

# ── Plot 1 — Raw count confusion matrix ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Grade Prediction Confusion Matrix', fontsize=14, fontweight='bold')

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=present_grades,
    yticklabels=present_grades,
    cmap='Blues',
    linewidths=0.5,
    ax=axes[0],
    cbar=True
)
axes[0].set_xlabel('Predicted Grade', fontsize=12)
axes[0].set_ylabel('Actual Grade',    fontsize=12)
axes[0].set_title('Raw Counts', fontsize=12, fontweight='bold')

# ── Plot 2 — Normalized confusion matrix ──────────────────────────────────────
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    xticklabels=present_grades,
    yticklabels=present_grades,
    cmap='YlOrRd',
    linewidths=0.5,
    ax=axes[1],
    vmin=0, vmax=1,
    cbar=True
)
axes[1].set_xlabel('Predicted Grade', fontsize=12)
axes[1].set_ylabel('Actual Grade',    fontsize=12)
axes[1].set_title('Normalized (Row = Recall per Grade)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../eda/eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/eval_confusion_matrix.png')

Plot saved → eda/eval_confusion_matrix.png


 Final Evaluation Report

Per Class Metrics from Confusion Matrix

In [15]:
from sklearn.metrics import classification_report

print('── Per Grade Classification Report ──')
print()
print(classification_report(
    eval_df['actual_grade'],
    eval_df['pred_grade'],
    labels=present_grades,
    digits=4,
    zero_division=0
))

# ── Per class breakdown from confusion matrix ─────────────────────────────────
print('── Per Grade Confusion Matrix Breakdown ──')
print()
print(f"{'Grade':<6} {'TP':>6} {'FP':>6} {'FN':>6} {'TN':>6} "
      f"{'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
print('-' * 75)

for i, grade in enumerate(present_grades):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - TP - FP - FN

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)
    support   = cm[i, :].sum()

    print(f"{grade:<6} {TP:>6} {FP:>6} {FN:>6} {TN:>6} "
          f"{precision:>10.4f} {recall:>8.4f} {f1:>8.4f} {support:>9}")

print()

# ── Overall summary ───────────────────────────────────────────────────────────
total_correct = np.trace(cm)
total_samples = cm.sum()
overall_acc   = total_correct / total_samples

print(f'Overall Accuracy   : {overall_acc:.4f}  ({total_correct}/{total_samples})')
print(f'Exact grade match  : {exact_match:.1%}')
print(f'Within 1 band      : {adjacent_match:.1%}')

── Per Grade Classification Report ──

              precision    recall  f1-score   support

           C     0.9333    0.5600    0.7000        25
          C+     0.6927    0.7089    0.7007       213
           B     0.7359    0.7918    0.7628       461
          B+     0.7273    0.6640    0.6942       253
           A     0.6250    0.5208    0.5682        48

    accuracy                         0.7230      1000
   macro avg     0.7428    0.6491    0.6852      1000
weighted avg     0.7241    0.7230    0.7213      1000

── Per Grade Confusion Matrix Breakdown ──

Grade      TP     FP     FN     TN  Precision   Recall       F1   Support
---------------------------------------------------------------------------
C          14      1     11    974     0.9333   0.5600   0.7000        25
C+        151     67     62    720     0.6927   0.7089   0.7007       213
B         365    131     96    408     0.7359   0.7918   0.7628       461
B+        168     63     85    684     0.7273   0.6640  

In [13]:
plots_saved = [
    'eda/eval_predicted_vs_actual.png',
    'eda/eval_residuals.png',
    'eda/eval_feature_importance.png',
    'eda/eval_error_by_grade_band.png',
    'eda/eval_risk_and_grade_distribution.png',
    'eda/eval_grade_agreement.png',
]

print('╔══════════════════════════════════════════════════════════╗')
print('║              MODEL EVALUATION REPORT                    ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Regression Metrics (Test Set)                          ║')
print(f'║    MAE  : {mae:.4f}   {"✅ target < 8"  if mae  < 8    else "⚠️  above target":<30}║')
print(f'║    RMSE : {rmse:.4f}   {"✅ target < 12" if rmse < 12   else "⚠️  above target":<30}║')
print(f'║    R²   : {r2:.4f}   {"✅ target ≥ 0.80" if r2 >= 0.80 else "⚠️  below target":<30}║')
print(f'║    MAPE : {mape:.2f}%                                        ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Grade Prediction Agreement                             ║')
print(f'║    Exact match   : {exact_match:.1%}                              ║')
print(f'║    Within 1 band : {adjacent_match:.1%}                              ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Early Warning System                                   ║')
print(f'║    Actual failures      : {total_actual_failures:<5}                       ║')
print(f'║    Caught (HIGH+CRIT)   : {caught_in_high_critical:<5}                       ║')
print(f'║    Early warning recall : {recall:.1%}                        ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Plots Saved:                                           ║')
for p in plots_saved:
    print(f'║    {p:<52}║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Next step → 08_model_export.ipynb                      ║')
print('╚══════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════╗
║              MODEL EVALUATION REPORT                    ║
╠══════════════════════════════════════════════════════════╣
║  Regression Metrics (Test Set)                          ║
║    MAE  : 2.5951   ✅ target < 8                  ║
║    RMSE : 3.2475   ✅ target < 12                 ║
║    R²   : 0.8437   ✅ target ≥ 0.80               ║
║    MAPE : 4.01%                                        ║
╠══════════════════════════════════════════════════════════╣
║  Grade Prediction Agreement                             ║
║    Exact match   : 72.3%                              ║
║    Within 1 band : 100.0%                              ║
╠══════════════════════════════════════════════════════════╣
║  Early Warning System                                   ║
║    Actual failures      : 238                         ║
║    Caught (HIGH+CRIT)   : 152                         ║
║    Early warning recall : 63.9%                        ║
╠═════